In [2]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Inisialisasi MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [3]:
def calculate_angle(a, b, c):
    a = np.array(a) # Titik 1 (misal: Panggul)
    b = np.array(b) # Titik 2 (misal: Lutut - Vertex)
    c = np.array(c) # Titik 3 (misal: Pergelangan Kaki)
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
    return angle

In [4]:
# Tentukan path video kamu (masukkan video contoh di folder data/)
video_path = '../data/video_sample.mp4' 
cap = cv2.VideoCapture(video_path)
data_list = []

print("Memulai ekstraksi fitur...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Konversi ke RGB
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image)
    
    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        
        # Ambil koordinat kunci (Contoh: Kaki Kanan)
        hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
        knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
        ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
        
        # Hitung sudut lutut
        knee_angle = calculate_angle(hip, knee, ankle)
        
        # Simpan semua 33 koordinat x,y,z + sudut ke dalam dictionary
        row = {"knee_angle": knee_angle}
        for i, lm in enumerate(landmarks):
            row[f'x{i}'] = lm.x
            row[f'y{i}'] = lm.y
            row[f'z{i}'] = lm.z
            row[f'v{i}'] = lm.visibility
            
        data_list.append(row)

cap.release()

# Simpan ke DataFrame
df = pd.DataFrame(data_list)
print(f"Berhasil mengekstrak {len(df)} frame!")
df.head()

Memulai ekstraksi fitur...
Berhasil mengekstrak 268 frame!


,knee_angle,x0,y0,z0,v0,x1,y1,z1,v1,x2,...,z30,v30,x31,y31,z31,v31,x32,y32,z32,v32
0,167.948904,0.725168,-0.157717,-0.095615,0.978616,0.719429,-0.172144,-0.079777,0.975320,0.719128,...,0.029831,0.921858,0.694681,0.662649,0.087600,0.978828,0.467657,0.683826,-0.080386,0.989551
1,156.006712,0.728066,-0.154602,-0.006425,0.979014,0.720805,-0.166031,0.001985,0.976032,0.719668,...,-0.099901,0.925428,0.728133,0.665985,0.095216,0.979044,0.445017,0.649674,-0.185306,0.990025
2,128.732488,0.719870,-0.153014,0.217266,0.981088,0.717296,-0.166317,0.212562,0.978396,0.715114,...,-0.148520,0.932170,0.741038,0.682208,0.095585,0.980181,0.431527,0.597426,-0.190521,0.990865
3,91.423395,0.725652,-0.055962,0.060278,0.977693,0.722295,-0.068930,0.075623,0.971767,0.719994,...,-0.169213,0.937320,0.732750,0.701746,0.133422,0.975661,0.440298,0.528786,-0.208824,0.991365
4,86.614815,0.729946,-0.077268,0.066833,0.979473,0.726071,-0.088740,0.084845,0.973921,0.725229,...,-0.175716,0.940016,0.668679,0.711121,0.041902,0.975671,0.493684,0.495983,-0.215224,0.991545


In [5]:
def label_data(angle):
    if angle < 165: # Contoh threshold overstride
        return "Overstride"
    else:
        return "Normal"

df['label'] = df['knee_angle'].apply(label_data)
df.to_csv('../data/running_dataset_cleaned.csv', index=False)
print("Dataset siap digunakan untuk training!")

Dataset siap digunakan untuk training!
